In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report
)

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "10s-30s"

TRAIN_FILE = DATA_DIR / "train_features.csv"
VALIDATION_FILE = DATA_DIR / "validation_features.csv"
TEST_FILE = DATA_DIR / "test_features.csv"

print("Project folder:", ROOT)
print("Feature folder:", DATA_DIR)
print("Train file exists:", TRAIN_FILE.exists())
print("Validation file exists:", VALIDATION_FILE.exists())
print("Test file exists:", TEST_FILE.exists())

In [ ]:
missing = [p for p in [TRAIN_FILE, VALIDATION_FILE, TEST_FILE] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing feature file(s): " + ", ".join(str(p) for p in missing) +
        ". The *_windows.csv metadata files are not substitutes for extracted feature tables."
    )

train = pd.read_csv(TRAIN_FILE)
validation = pd.read_csv(VALIDATION_FILE)
test = pd.read_csv(TEST_FILE)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)
display(train.head())

In [ ]:
FEATURES = [
    "sig_mean", "sig_std", "sig_rms", "sig_min", "sig_max", "sig_ptp",
    "line_length", "n_beats", "mean_rr", "sdnn", "rr_cv", "rmssd",
    "med_abs_drr", "mean_hr", "pnn50"
]
TARGET = "label"

for name, df in [("train", train), ("validation", validation), ("test", test)]:
    if TARGET not in df.columns:
        raise ValueError(f"{name} is missing '{TARGET}'.")
    missing_features = [c for c in FEATURES if c not in df.columns]
    if missing_features:
        raise ValueError(f"{name} is missing expected features: {missing_features}")
    print(name, "label values:", sorted(df[TARGET].dropna().unique()))

X_train = train[FEATURES].apply(pd.to_numeric, errors="coerce")
X_val = validation[FEATURES].apply(pd.to_numeric, errors="coerce")
X_test = test[FEATURES].apply(pd.to_numeric, errors="coerce")

y_train = train[TARGET].astype(int)
y_val = validation[TARGET].astype(int)
y_test = test[TARGET].astype(int)

print("\nTraining label counts:")
display(y_train.value_counts().sort_index().to_frame("count"))
print("Positive prevalence — train:", round(y_train.mean(), 4))
print("Positive prevalence — validation:", round(y_val.mean(), 4))
print("Positive prevalence — test:", round(y_test.mean(), 4))

In [ ]:
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ))
])
logistic_model.fit(X_train, y_train)

logistic_val_prob = logistic_model.predict_proba(X_val)[:, 1]
logistic_val_pred = (logistic_val_prob >= 0.5).astype(int)
print("Logistic Regression trained.")

In [ ]:
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=20,
        class_weight="balanced_subsample", random_state=42, n_jobs=-1
    ))
])
rf_model.fit(X_train, y_train)

rf_val_prob = rf_model.predict_proba(X_val)[:, 1]
rf_val_pred = (rf_val_prob >= 0.5).astype(int)
print("Random Forest trained.")

In [ ]:
def get_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "Brier score": brier_score_loss(y_true, y_prob)
    }

validation_results = pd.DataFrame({
    "Logistic Regression": get_metrics(y_val, logistic_val_pred, logistic_val_prob),
    "Random Forest": get_metrics(y_val, rf_val_pred, rf_val_prob)
}).T

display(validation_results.round(4))

In [ ]:
print("Logistic Regression — validation")
print("Confusion matrix:\n", confusion_matrix(y_val, logistic_val_pred))
print(classification_report(y_val, logistic_val_pred, zero_division=0))

print("\nRandom Forest — validation")
print("Confusion matrix:\n", confusion_matrix(y_val, rf_val_pred))
print(classification_report(y_val, rf_val_pred, zero_division=0))

In [ ]:
SELECTED_MODEL = "logistic"  # Set to "random_forest" if selected on validation data.

if SELECTED_MODEL == "logistic":
    final_model = logistic_model
elif SELECTED_MODEL == "random_forest":
    final_model = rf_model
else:
    raise ValueError("Choose 'logistic' or 'random_forest'.")

test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

test_results = pd.DataFrame(
    [get_metrics(y_test, test_pred, test_prob)], index=[SELECTED_MODEL]
)
display(test_results.round(4))
print("Test confusion matrix:\n", confusion_matrix(y_test, test_pred))
print(classification_report(y_test, test_pred, zero_division=0))